In [1]:
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# ============================================================
# 1. Load your LLM
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)



config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [2]:

# ============================================================
# 2. Define tools
# ============================================================

def calculator(expression):
    """
    Simple calculator tool.
    """
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


def read_file(path):
    """
    Read a local text file.
    """
    try:
        with open(path, "r") as f:
            return f.read()
    except Exception as e:
        return f"Error: {e}"


TOOLS = {
    "calculator": calculator,
    "read_file": read_file
}



In [3]:

# ============================================================
# 3. Tell the LLM about available tools
# ============================================================

SYSTEM_PROMPT = """
You are an AI agent.

You have access to these tools:

1. calculator
   Input:
   {"expression": "25 * 40"}

2. read_file
   Input:
   {"path": "test.py"}

If you need a tool, respond ONLY in this format:

TOOL:
{
    "name": "calculator",
    "arguments": {
        "expression": "25 * 40"
    }
}

or:

TOOL:
{
    "name": "read_file",
    "arguments": {
        "path": "test.py"
    }
}

If you don't need a tool, answer normally.
"""


In [4]:


# ============================================================
# 4. LLM function
# ============================================================

def ask_llm(prompt):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response



In [5]:

# ============================================================
# 5. Agent loop
# ============================================================

def agent(user_prompt):

    response = ask_llm(user_prompt)

    print("\nLLM:")
    print(response)

    # Check whether LLM requested a tool
    if response.startswith("TOOL:"):

        tool_json = response.replace("TOOL:", "").strip()

        request = json.loads(tool_json)

        tool_name = request["name"]
        arguments = request["arguments"]

        print("\nExecuting tool:", tool_name)

        # Execute requested tool
        result = TOOLS[tool_name](**arguments)

        print("\nTool result:")
        print(result)

        # Send result back to LLM
        final_prompt = f"""
User request:

{user_prompt}

You requested the tool:

{tool_name}

The tool returned:

{result}

Now provide the final answer to the user.
"""

        final_response = ask_llm(final_prompt)

        print("\nFinal answer:")
        print(final_response)

    else:

        print("\nFinal answer:")
        print(response)



In [6]:

# ============================================================
# 6. Test agent
# ============================================================

agent("What is 25 * 40?")


LLM:
TOOL:
{
    "name": "calculator",
    "arguments": {
        "expression": "25 * 40"
    }
}

Executing tool: calculator

Tool result:
1000

Final answer:
The result of 25 * 40 is 1000.


In [11]:
agent("read the file /content/requirements.txt, any discripbe why we need those content")


LLM:
TOOL:
{
    "name": "read_file",
    "arguments": {
        "path": "/content/requirements.txt"
    }
}

Executing tool: read_file

Tool result:
torch >= 2.0.1 
jupyterlab >= 4.0
tiktoken >= 0.5.1
matplotlib >= 3.7.1
numpy >= 1.24.3
tensorflow >= 2.15.0
tqdm >= 4.66.1
numpy >= 1.25, < 2.0
pandas >= 2.2.1
psutil >= 5.9.5
litgpt[all] >= 0.4.1

Final answer:
The `/content/requirements.txt` file lists the minimum required versions of various Python packages that are needed for a project or environment. Here's a brief explanation of each package and its version requirement:

- **torch >= 2.0.1**: This is a deep learning library developed by Facebook's AI Research lab. Version 2.0.1 is the minimum required version.
- **jupyterlab >= 4.0**: JupyterLab is an interactive development environment for Jupyter Notebooks. Version 4.0 is the minimum required version.
- **tiktoken >= 0.5.1**: Tiktoken is a library used for tokenizing text. Version 0.5.1 is the minimum required version.
- **matpl